# Q1 Lindblad in the Interaction Picture under Time-Varying Ez

Custom Lindblad solve for `Q1_F1_3o2_F1` under the same `210 → 200 → 210 V/cm` quadratic Ez(t)
trajectory used by `q1_effective_fixed_basis_vs_static_regular_rust.ipynb`, but **without** any
effective-Hamiltonian interpolation.

**Approach.** Diagonalize the molecular Hamiltonian once at Ez=200 V/cm to get the dressed basis V_ref.
Stark mixing of J=0,2,3 *into* the dressed J=1 manifold is captured exactly by `eigh`. Move to the
interaction picture by absorbing `diag(E_ref)` analytically into `U(t) = exp(−i diag(E_ref) t)` —
this removes the diagonal-stiffness (~80 GHz from far-detuned dressed states) that would otherwise force
an explicit ODE solver to take ~ps step sizes despite the dressed dynamics being µs-scale.

Time-varying Ez(t) enters as a slowly-varying perturbation `δH(t) = (Ez(t) − 200) · dHSz_dressed` in
the V_ref basis. A secular RWA mask drops H entries with `|E_α − E_β| > 1 GHz` (cross-manifold
couplings whose physical effect is < 10 Hz AC-Stark and so safely neglected); within-manifold MHz-scale
Stark mixing is retained. The dissipator uses the fully-secular Lindblad form so cross-coherence terms
in the (B,B) block don't drive populations — eliminates fast-oscillating dissipator products.

**Bare basis.** X uncoupled J=0..3 (64 states) and B coupled-Ω J=1..3 with both Ω signs (120 states),
so opposite-parity B states are present and Λ-doubling is a static off-diagonal in the matrix. Total
184 states. With this scheme RK45 takes µs steps and the full 50 µs dynamic run completes in ~12 min
with trace conservation at machine precision.

**Comparison.** Plotted side-by-side against the effective fixed-basis interpolated model from the
existing notebook at the bottom.


In [ ]:
from pathlib import Path
import sys
import time
import warnings


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "centrex_tlf").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the notebook working directory.")


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

warnings.filterwarnings(
    "ignore",
    message=r"Low overlap detected for approximate states",
    category=UserWarning,
)

import matplotlib.pyplot as plt
import numpy as np
import sympy as smp
from scipy.integrate import solve_ivp

from centrex_tlf import couplings, hamiltonian, lindblad, states, transitions
from centrex_tlf.hamiltonian.basis_transformations import generate_transform_matrix
from centrex_tlf.hamiltonian.matrix_elements_electric_dipole import ED_ME_coupled
from centrex_tlf.states.generate_states import generate_coupled_states_B
from centrex_tlf.states.utils import reorder_evecs

from centrex_tlf.effective_hamiltonian import (
    prepare_effective_lindblad_rust_plan,
    prepare_lindblad_safe_compact_interpolated_model,
    solve_effective_lindblad,
)
from centrex_tlf.effective_hamiltonian.compact_reference import (
    build_compact_reference_decomposed_bundle,
)
from centrex_tlf.lindblad.parameters import LindbladParameters, Parameter, Time

plt.rcParams.update({"axes.grid": True, "font.size": 14})

## Configuration

Mirror the parameters of `q1_effective_fixed_basis_vs_static_regular_rust.ipynb`.

In [ ]:
transition = transitions.Q1_F1_3o2_F1
polarization = couplings.polarization_Z

Gamma = 2.0 * np.pi * 1.56e6                 # rad/s
Omega0 = Gamma                               # peak Rabi (rad/s)
velocity = 184.0                             # m/s
t_total = 50e-6                              # s
detuning_at_200 = 0.0                        # rad/s, referenced to dressed Q1 transition at Ez=200

E_static = 200.0                             # V/cm (laser/rotating-frame reference)
E_edge = 210.0                               # V/cm (trajectory edges)
magnetic_field = np.array([0.0, 0.0, 1e-5])  # G
field_points = np.linspace(E_static, E_edge, 11)
t_eval = np.linspace(0.0, t_total, 251)

ode_method = "RK45"   # Switch to 'LSODA' or 'BDF' if Λ-induced stiffness slows things down.
ode_rtol = 1e-7
ode_atol = 1e-9

# Same quadratic Ez(t) profile as the reference notebook (210 -> 200 at midpoint -> 210)
_half_span = 0.5 * velocity * t_total


def Ez_of_t(t):
    z = velocity * (t - 0.5 * t_total)
    return E_static + (E_edge - E_static) * (z / _half_span) ** 2


def dEz_dt(t):
    z = velocity * (t - 0.5 * t_total)
    return (E_edge - E_static) * 2.0 * z * velocity / _half_span ** 2


Ez_profile = np.vectorize(Ez_of_t)(t_eval)
dEz_profile = np.vectorize(dEz_dt)(t_eval)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
axes[0].plot(t_eval * 1e6, Ez_profile)
axes[0].axhline(E_static, color="k", ls="--", lw=1)
axes[0].set(xlabel="t (us)", ylabel="Ez (V/cm)", title="Quadratic Ez(t)")
axes[1].plot(t_eval * 1e6, dEz_profile / 1e6)
axes[1].set(xlabel="t (us)", ylabel="dEz/dt (V/cm / us)", title="dEz/dt")
plt.tight_layout()
plt.show()

print(f"Transition: {transition}")
print(f"Gamma / (2π): {Gamma / (2*np.pi):.3e} Hz")

## Bare basis and field-independent molecular operators

X side: build in the **uncoupled** basis (where the package's `HamiltonianUncoupledX` blocks live), then transform to the coupled basis so X and B share a hyperfine-coupled representation. This makes the X<->B optical dipole projection a clean coupled-basis matrix product.

B side: coupled-Ω basis with both Ω=±1 (so opposite-parity states are present and Λ-doubling is a static off-diagonal in the matrix).

In [ ]:
# X uncoupled J=0..3 (64 states); transform to coupled basis
QN_X_uc = list(states.generate_uncoupled_states_ground(Js=np.arange(0, 4)))
QN_X = list(states.generate_coupled_states_ground(Js=np.arange(0, 4)))
S_X = generate_transform_matrix(QN_X_uc, QN_X)            # uncoupled -> coupled (64 x 64)

# X Hamiltonian pieces (uncoupled). 2π scaling is applied inside the assembly function below.
_HX = hamiltonian.generate_uncoupled_hamiltonian_X(QN_X_uc)


def _build_HX_coupled(Ez):
    """Total X Hamiltonian in the COUPLED basis at field (0, 0, Ez), B = magnetic_field. rad/s."""
    Bx, By, Bz = magnetic_field
    HX_uc = 2 * np.pi * (
        _HX.Hff
        + Ez * _HX.HSz
        + Bx * _HX.HZx + By * _HX.HZy + Bz * _HX.HZz
    )
    return S_X.conj().T @ HX_uc @ S_X


# d/dEz of HX in the coupled basis (precomputed; used for Λ_X numerator)
dHX_dEz_coupled = 2 * np.pi * (S_X.conj().T @ _HX.HSz @ S_X)

# B coupled-Omega J=1..3, both Omega (120 states)
QN_B = list(generate_coupled_states_B(
    states.QuantumSelector(J=np.arange(1, 4), P=None, Ω=[-1, 1])
))
_HB = hamiltonian.generate_coupled_hamiltonian_B(QN_B)


def _build_HB(Ez):
    """Total B Hamiltonian in the coupled-Ω basis at (0, 0, Ez). rad/s."""
    Bx, By, Bz = magnetic_field
    return 2 * np.pi * (
        _HB.Hrot + _HB.H_mhf_Tl + _HB.H_mhf_F + _HB.H_LD + _HB.H_cp1_Tl + _HB.H_c_Tl
        + Ez * _HB.HSz
        + Bx * _HB.HZx + By * _HB.HZy + Bz * _HB.HZz
    )


dHB_dEz = 2 * np.pi * _HB.HSz

n_X = len(QN_X)   # 64
n_B = len(QN_B)   # 120
n_tot = n_X + n_B  # 184
print(f"Bare basis: n_X={n_X}, n_B={n_B}, n_tot={n_tot}")

## Bare-basis dipole matrices D_q (q ∈ {-1, 0, +1}) and laser-polarization dipole D_L

`D_q[i, j] = ⟨X_coupled,i | d_q | B_coupled-Ω,j⟩` for q ∈ {-1, 0, +1} (spherical components). Used to build:

- The **optical coupling block** under the laser polarization (`D_L`): only the q matching the laser drives the rotating-frame Rabi term.
- The **decay jump operators** `C_q = √Γ · D_q^bare` (one per polarization channel; total decay rate per pure-B-J=1 state is Γ by the sum rule we verified).

The polarization vector is the laser polarization in Cartesian (Ex, Ey, Ez); for `polarization_Z` the laser is π (q=0).

In [ ]:
# Spherical-component polarization vectors (Cartesian) used for ED_ME_coupled.
# These select a single q via the angular_part(): q = mF_bra - mF_ket.
POL_q = {
    -1: (1.0 / np.sqrt(2) + 0j, -1j / np.sqrt(2), 0 + 0j),  # σ-
    0:  (0 + 0j, 0 + 0j, 1.0 + 0j),                         # π
    +1: (-1.0 / np.sqrt(2) + 0j, -1j / np.sqrt(2), 0 + 0j),  # σ+
}


def _build_dipole(QN_X, QN_B, pol_vec):
    D = np.zeros((len(QN_X), len(QN_B)), dtype=complex)
    for i, qx in enumerate(QN_X):
        for j, qb in enumerate(QN_B):
            D[i, j] = ED_ME_coupled(qx, qb, pol_vec=tuple(pol_vec))
    return D


Dq_bare = {q: _build_dipole(QN_X, QN_B, POL_q[q]) for q in (-1, 0, 1)}

# Sanity: per-B-state total |d|^2 should be 1.0 for J'=1 states (covered by X J=0..2)
tot_dsq = sum(np.abs(Dq_bare[q]) ** 2 for q in (-1, 0, 1)).sum(axis=0)
j1_idx = np.array([qb.J == 1 for qb in QN_B])
assert np.allclose(tot_dsq[j1_idx], 1.0, atol=1e-10), "Sum rule on J'=1 dipole strengths broken"
print(f"Sum rule OK: |d|^2 totals on J'=1 = {tot_dsq[j1_idx].mean():.6f}")

# Laser-polarization dipole. polarization_Z.vector is Cartesian [0, 0, 1] -> drives only q=0.
pol_cart = tuple(polarization.vector)
D_L_bare = _build_dipole(QN_X, QN_B, pol_cart)  # n_X x n_B
print(f"D_L_bare shape={D_L_bare.shape}, max|D_L|={np.abs(D_L_bare).max():.4f}")

## Reference dressed basis at Ez = 200 V/cm and rotating-frame anchor ω_L

Diagonalize H_X(200) and H_B(200) once, fix the gauge by overlap-tracking against the identity, and use the resulting eigenvectors as `V_X_ref`, `V_B_ref`. These serve as the reference for `reorder_evecs` at every subsequent RHS call so the dressed-state index for a given physical state is preserved across t.

The rotating-frame anchor `ω_L` is set to the **strongest dipole-coupled** dressed pair within (X J=1, B J=1 F1'=3/2): we compute the projected dressed-basis dipole, find argmax over the resonant block, and place that pair on resonance (matching the existing OBE convention which puts the named-transition main coupling at zero detuning). Anchoring on the mean instead would put every dressed pair 100–250 MHz off resonance because the F1'=3/2 J=1 manifold is parity-mixed by Stark and spans ~440 MHz at Ez=200 V/cm.


In [ ]:
def diag_with_ref(H, V_ref):
    """Diagonalize H, reorder eigenvectors so they best-overlap with V_ref columns.
    Returns (E, V) with V column k aligned to V_ref column k."""
    E, V = np.linalg.eigh(H)
    E_sorted, V_sorted = reorder_evecs(V, E, V_ref)
    # reorder_evecs leaves global phase unfixed; lock it by aligning the largest-magnitude
    # component of each new eigenvector to a real-positive value (gauge fix).
    for k in range(V_sorted.shape[1]):
        idx = np.argmax(np.abs(V_sorted[:, k]))
        ph = V_sorted[idx, k]
        if np.abs(ph) > 0:
            V_sorted[:, k] = V_sorted[:, k] * np.conj(ph) / np.abs(ph)
    return E_sorted, V_sorted


HX_ref = _build_HX_coupled(E_static)
HB_ref = _build_HB(E_static)
E_X_ref, V_X_ref = diag_with_ref(HX_ref, np.eye(n_X, dtype=complex))
E_B_ref, V_B_ref = diag_with_ref(HB_ref, np.eye(n_B, dtype=complex))

# Identify dressed-state indices by dominant bare-component (J, F1) character.
_X_J_of_bare  = np.array([qb.J for qb in QN_X])
_X_F1_of_bare = np.array([float(qb.F1) for qb in QN_X])
_B_J_of_bare  = np.array([qb.J for qb in QN_B])
_B_F1_of_bare = np.array([float(qb.F1) for qb in QN_B])

_dom_bare_X = np.argmax(np.abs(V_X_ref) ** 2, axis=0)
_dom_bare_B = np.argmax(np.abs(V_B_ref) ** 2, axis=0)
X_J_of_dressed  = _X_J_of_bare[_dom_bare_X]
X_F1_of_dressed = _X_F1_of_bare[_dom_bare_X]
B_J_of_dressed  = _B_J_of_bare[_dom_bare_B]
B_F1_of_dressed = _B_F1_of_bare[_dom_bare_B]

idx_X_J1        = np.where(X_J_of_dressed == 1)[0]                                          # 12 dressed X J=1
idx_B_J1        = np.where(B_J_of_dressed == 1)[0]                                          # 24 dressed B J=1
idx_B_J1_F1_3o2 = np.where((B_J_of_dressed == 1) & (np.isclose(B_F1_of_dressed, 1.5)))[0]    # 16 dressed B F1'=3/2 J=1

# Pre-project the laser-polarization dipole into the dressed basis (raw, no normalization yet).
_D_L_dressed_raw = V_X_ref.conj().T @ D_L_bare @ V_B_ref                                    # n_X x n_B

# Anchor omega_L on the strongest dipole-coupled pair in the resonant block (X J=1, B F1'=3/2 J=1).
# This matches the existing OBE machinery, which puts the named-transition main coupling at zero detuning.
_sub = _D_L_dressed_raw[np.ix_(idx_X_J1, idx_B_J1_F1_3o2)]
_ai_max, _bi_max = np.unravel_index(np.argmax(np.abs(_sub)), _sub.shape)
_a_strongest = idx_X_J1[_ai_max]; _b_strongest = idx_B_J1_F1_3o2[_bi_max]
peak_dipole_resonant = float(np.abs(_sub).max())
omega_L = float(E_B_ref[_b_strongest] - E_X_ref[_a_strongest]) + detuning_at_200

print(f"Dressed X J=1 count: {len(idx_X_J1)}, B J=1 count: {len(idx_B_J1)}, B J=1 F1'=3/2 count: {len(idx_B_J1_F1_3o2)}")
print(f"Strongest pair: X dressed[{_a_strongest}] <-> B dressed[{_b_strongest}], raw dipole = {peak_dipole_resonant:.4f}")
print(f"omega_L / (2π) = {omega_L / (2*np.pi):.3e} Hz   (anchored on strongest pair)")


## Initial state: uniform population on dressed X J=1 manifold

Match the existing notebook's `uniform_density_from_selector(QuantumSelector(J=1, electronic=X))` semantics: uniform diagonal population on the 12 dressed X J=1 states, total trace = 1, all coherences zero. The full density matrix lives in the 184-dimensional dressed basis (block-diagonal in X | B at t=0).

In [ ]:
rho0 = np.zeros((n_tot, n_tot), dtype=complex)
rho0[idx_X_J1, idx_X_J1] = 1.0 / len(idx_X_J1)
assert np.isclose(np.trace(rho0).real, 1.0)
print(f"rho0 diagonal mass on dressed X J=1: {rho0[idx_X_J1, idx_X_J1].sum().real:.6f}")

## Interaction-picture RHS (V_ref basis fixed at Ez=200, secular RWA)

We absorb the diagonal of the dressed Hamiltonian analytically into the unitary `U(t) = exp(-i diag(E_ref) t)`.
The integrated state is `ρ_int = U† ρ U`; only off-diagonals of H_residual contribute, multiplied by
phase factors `exp(i (E_α - E_β) t)`. A secular **RWA mask** zeros entries with `|E_α - E_β| > cutoff`
(default 1 GHz): kept entries are within-manifold (MHz-scale, including the within-F1=3/2 Stark mixing); dropped
entries are cross-manifold (~14 GHz hyperfine gap, ~13 GHz J–J gap) whose physical effect on the resonant
manifold is < ~10 Hz AC Stark and so safe to neglect.

**Optical normalization.** `H_optical_const` is normalized so its peak entry within the resonant
block equals `Omega0/2` — i.e. `Omega0` is the peak Rabi between the strongest dipole-coupled
dressed pair, matching the existing OBE machinery's `normalize_pol=True` convention. The decay
operators `C_q_const` are NOT normalized — they carry the physical √Γ factor and the bare-basis
dipole sum rule guarantees per-B-state total decay = Γ for J=1 F1'=3/2 dressed states.

The dissipator is in **fully-secular Lindblad form**: each (X_dressed, B_dressed) bare-decay channel contributes
an incoherent population transfer rate `|C_q[α,β]|^2`; cross-coherences in the (B,B) block don't get mixed into
X populations. This eliminates fast-oscillating dissipator products.

Because the V_ref basis is fixed (chosen at Ez=200 V/cm), there is **no moving-basis term** in this formulation.
Time-varying Ez(t) enters via `δH(t) = (Ez(t) − 200) · dHSz_dressed`, a small slowly-varying perturbation in V_ref
basis (within-manifold component drives slow detuning shifts; cross-manifold component is RWA-suppressed). The
Stark mixing of J=0,2,3 *into* the dressed J=1 states is captured **exactly** by the eigh at Ez=200, not eliminated.

Why this works numerically: removing the diagonal energies (which span ~80 GHz across the bare basis) from the
explicit integration eliminates the stiffness that forced ~ps step sizes. After the IP transform + RWA, RK45
takes ~µs-scale steps. Trace conservation remains at machine precision.


In [ ]:
X_sl = slice(0, n_X)
B_sl = slice(n_X, n_tot)

# Concatenated dressed-basis energies in the rotating frame on B (B states shifted by -omega_L).
E_diag_ref = np.concatenate([E_X_ref, E_B_ref - omega_L])

# Optical block normalized so peak Rabi within the resonant block = Omega0 (matches existing OBE).
H_optical_const = (Omega0 / 2.0) * (V_X_ref.conj().T @ D_L_bare @ V_B_ref) / peak_dipole_resonant
print(f"Peak |H_optical_const| / (2π) = {np.abs(H_optical_const).max() / (2 * np.pi) / 1e6:.3f} MHz "
      f"(target Omega0/(4π) = {Omega0 / (4 * np.pi) / 1e6:.3f} MHz)")

# Decay operators carry the physical sqrt(Gamma) factor (NOT normalized).
C_q_const = {q: np.sqrt(Gamma) * (V_X_ref.conj().T @ Dq_bare[q] @ V_B_ref) for q in (-1, 0, 1)}
dHX_dEz_dressed = V_X_ref.conj().T @ dHX_dEz_coupled @ V_X_ref
dHB_dEz_dressed = V_B_ref.conj().T @ dHB_dEz @ V_B_ref

# Secular RWA mask: keep matrix entries with dressed-energy gap < cutoff (default 1 GHz)
RWA_cutoff = 1.0e9 * 2 * np.pi  # rad/s
_dE = E_diag_ref[:, None] - E_diag_ref[None, :]
RWA_mask = np.abs(_dE) < RWA_cutoff
print(f"RWA mask: {RWA_mask.sum()}/{n_tot**2} entries kept ({100*RWA_mask.sum()/n_tot**2:.1f}%)")

# Per-(X<-B) total decay rate (sum over q) for the secular dissipator
total_decay_rate = sum(np.abs(C_q_const[q]) ** 2 for q in (-1, 0, 1))   # n_X x n_B
total_decay_per_B = total_decay_rate.sum(axis=0)                        # n_B (= Gamma for J=1 F1'=3/2 by sum rule)

half_span_z = 0.5 * velocity * t_total


def Ez_of_t(t):
    z = velocity * (t - 0.5 * t_total)
    return E_static + (E_edge - E_static) * (z / half_span_z) ** 2


def make_rhs(constant_Ez=None):
    """Interaction-picture RHS in V_ref basis. Pass constant_Ez to a numeric value to disable
    the time-varying Ez(t) profile (used for the static sanity check)."""

    def rhs(t, rho_flat):
        rho = rho_flat.reshape(n_tot, n_tot)

        Ez = constant_Ez if constant_Ez is not None else Ez_of_t(t)
        delta_Ez = Ez - E_static

        # H_lab in V_ref basis: diag(E_ref) + delta_Ez * dH/dEz_dressed + optical block
        H_lab = np.zeros((n_tot, n_tot), dtype=complex)
        H_lab[X_sl, X_sl] = np.diag(E_X_ref) + delta_Ez * dHX_dEz_dressed
        H_lab[B_sl, B_sl] = np.diag(E_B_ref - omega_L) + delta_Ez * dHB_dEz_dressed
        H_lab[X_sl, B_sl] = H_optical_const
        H_lab[B_sl, X_sl] = H_optical_const.conj().T

        # Subtract the diagonal absorbed analytically into U(t); rotate by exp phases
        H_residual = H_lab - np.diag(E_diag_ref)
        p = np.exp(1j * E_diag_ref * t)
        phase = np.outer(p, np.conj(p))           # phase[a,b] = exp(i (E_a - E_b) t)
        H_int = H_residual * phase * RWA_mask     # secular RWA on the Hamiltonian

        drho = -1j * (H_int @ rho - rho @ H_int)

        # Fully-secular dissipator: rate-equation population transfer + linear damping of B coherences
        rho_BB_diag = rho[B_sl, B_sl].diagonal()
        gain_X = total_decay_rate @ rho_BB_diag
        for i in range(n_X):
            drho[i, i] += gain_X[i]
        gamma_vec = np.zeros(n_tot)
        gamma_vec[B_sl] = total_decay_per_B
        damping = 0.5 * (gamma_vec[:, None] + gamma_vec[None, :])
        drho -= damping * rho

        return drho.ravel()

    return rhs


# Smoke test
_t0 = time.time()
_drho = make_rhs()(0.0, rho0.ravel())
print(f"Single RHS call: {(time.time() - _t0) * 1000:.1f} ms; |drho/dt|_max = {np.abs(_drho).max():.3e}")


## Sanity check: static Ez = 200 V/cm

With constant Ez, `δH(t) = 0` so only the constant optical block drives dynamics. This is the
regular static OBE (in the V_ref dressed basis) and serves as a baseline for trace conservation
and the dissipator normalization. Expect roughly ~17 min wall time for the full 50 µs at this basis size.


In [ ]:
rhs_static = make_rhs(constant_Ez=E_static)

_t = time.time()
sol_static = solve_ivp(
    rhs_static, (0.0, t_total), rho0.ravel(),
    t_eval=t_eval, method=ode_method, rtol=ode_rtol, atol=ode_atol,
)
print(f"static run: {time.time()-_t:.1f} s, status={sol_static.status}, msg={sol_static.message}")

rho_static = sol_static.y.T.reshape(-1, n_tot, n_tot)
trace_drift = np.max(np.abs(np.einsum('tii->t', rho_static).real - 1.0))
herm_drift = np.max(np.abs(rho_static - np.transpose(rho_static.conj(), (0, 2, 1))))
print(f"trace drift: {trace_drift:.3e}, hermiticity drift: {herm_drift:.3e}")

exc_pop_static = rho_static[:, B_sl, B_sl].diagonal(axis1=1, axis2=2).sum(axis=1).real
print(f"final excited (B) population, static Ez=200: {exc_pop_static[-1]:.6e}")

## Time-varying Ez run

Same Ez(t) trajectory as the existing notebook (210 → 200 → 210 V/cm quadratic over 50 µs). Fixed laser
frequency = `ω_L` defined at Ez=200 V/cm; as Ez varies the dressed transition shifts, manifesting as a
time-dependent detuning on the diagonal of `δH(t)` in the rotating frame. Wall time ~12 min.


In [ ]:
rhs_dyn = make_rhs()

_t = time.time()
sol_dyn = solve_ivp(
    rhs_dyn, (0.0, t_total), rho0.ravel(),
    t_eval=t_eval, method=ode_method, rtol=ode_rtol, atol=ode_atol,
)
print(f"dynamic run: {time.time()-_t:.1f} s, status={sol_dyn.status}, msg={sol_dyn.message}")

rho_dyn = sol_dyn.y.T.reshape(-1, n_tot, n_tot)
trace_drift_dyn = np.max(np.abs(np.einsum('tii->t', rho_dyn).real - 1.0))
herm_drift_dyn = np.max(np.abs(rho_dyn - np.transpose(rho_dyn.conj(), (0, 2, 1))))
print(f"trace drift: {trace_drift_dyn:.3e}, hermiticity drift: {herm_drift_dyn:.3e}")

## Diagnostics: excited population, scattering rate, total photons

Total scattering rate at time t = Σ_q tr(ρ · C_q†(t) C_q(t)) = Σ_q tr(ρ_BB · (V_B† D_q† V_X)(V_X† D_q V_B)) · Γ. Cumulative photons via trapezoidal integration.

In [ ]:
def excited_population(rho_t):
    return rho_t[:, B_sl, B_sl].diagonal(axis1=1, axis2=2).sum(axis=1).real


def scattering_rate(rho_t):
    """Sum_q tr(rho * C_q† C_q). C_q's are constant in V_ref basis (per bare-decay channel)."""
    CdC = np.zeros((n_B, n_B), dtype=complex)
    for q in (-1, 0, 1):
        CdC += C_q_const[q].conj().T @ C_q_const[q]
    return np.array([np.real(np.trace(rho[B_sl, B_sl] @ CdC)) for rho in rho_t])


def cumulative_integral(x, y):
    out = np.zeros_like(y, dtype=float)
    out[1:] = np.cumsum(0.5 * (y[1:] + y[:-1]) * np.diff(x))
    return out


exc_pop_dyn    = excited_population(rho_dyn)
scat_dyn       = scattering_rate(rho_dyn)
photons_dyn    = cumulative_integral(t_eval, scat_dyn)

exc_pop_static = excited_population(rho_static)
scat_static    = scattering_rate(rho_static)
photons_static = cumulative_integral(t_eval, scat_static)

print(f"dynamic final excited: {exc_pop_dyn[-1]:.6e}")
print(f"dynamic total photons: {photons_dyn[-1]:.6e}")
print(f"static  final excited: {exc_pop_static[-1]:.6e}")
print(f"static  total photons: {photons_static[-1]:.6e}")


## Comparison against the effective fixed-basis interpolated model

Re-run the effective branch from `q1_effective_fixed_basis_vs_static_regular_rust.ipynb` cell 4 with the same parameters and overlay.

In [ ]:
# Build the effective field expression with the IR-style Time/Parameter symbols
_t_sym = Time()
_v_p = Parameter("v", velocity)
_T_p = Parameter("t_total", t_total)
_z_expr = _v_p * (_t_sym - 0.5 * _T_p)
_half = 0.5 * _v_p * _T_p
Ez_expr_sym = E_static + (E_edge - E_static) * (_z_expr / _half) ** 2

eff_ref_system, _ = build_compact_reference_decomposed_bundle(
    transition=transition,
    optical_polarization=polarization,
    electric_field=(0.0, 0.0, E_static),
    magnetic_field=magnetic_field,
)
rabi_symbol = eff_ref_system.coupling_symbols[0]
detuning_symbol = next(
    s for s in eff_ref_system.H_symbolic.free_symbols
    if str(s).startswith("\u03b4") or str(s).lower().startswith("delta")
)

eff_model = prepare_lindblad_safe_compact_interpolated_model(
    field_points=field_points,
    transition=transition,
    optical_polarization=polarization,
    magnetic_field=magnetic_field,
    master_field=E_static,
)
rho0_eff = np.zeros((eff_model.n_effective_states, eff_model.n_effective_states), dtype=complex)
rho0_eff[np.asarray(eff_model.ground_indices), np.asarray(eff_model.ground_indices)] = 1.0
rho0_eff /= np.trace(rho0_eff)

eff_params = LindbladParameters({
    smp.Symbol("Ez"): Ez_expr_sym,
    rabi_symbol: Parameter("omega_rabi", Omega0),
    detuning_symbol: Parameter("delta_ref", detuning_at_200),
})

eff_plan = prepare_effective_lindblad_rust_plan(eff_model, eff_params)
eff_result = solve_effective_lindblad(
    eff_plan, rho0_eff, (0.0, t_total),
    saveat=t_eval, solver="dopri5", reltol=1e-7, abstol=1e-9, dt=1e-10, output="full",
)

eff_rho = eff_result.density_matrices()
eff_pop = eff_result.populations()
exc_pop_eff = eff_pop[:, eff_model.excited_indices].sum(axis=1)
scat_eff = np.array([
    np.real(np.trace(
        eff_rho[k] @ eff_model.effective_bundle((0.0, 0.0, float(Ez_profile[k]))).jump_rate_operator()
    ))
    for k in range(len(t_eval))
])
photons_eff = cumulative_integral(t_eval, scat_eff)
print(f"effective final excited: {exc_pop_eff[-1]:.6e}")
print(f"effective total photons: {photons_eff[-1]:.6e}")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 14), sharex=True)

axes[0].plot(t_eval * 1e6, Ez_profile, color="tab:blue")
axes[0].axhline(E_static, color="k", ls="--", lw=1)
axes[0].set_ylabel("Ez (V/cm)")
axes[0].set_title("Field profile and solver outputs")

axes[1].plot(t_eval * 1e6, exc_pop_dyn, lw=2, label="adiabatic dressed-frame (this notebook)")
axes[1].plot(t_eval * 1e6, exc_pop_eff, lw=2, ls="--", label="effective fixed-basis interpolated")
axes[1].plot(t_eval * 1e6, exc_pop_static, lw=1, alpha=0.6, label="adiabatic, static Ez=200 (sanity)")
axes[1].set_ylabel("total excited population")
axes[1].legend(fontsize=10)

axes[2].plot(t_eval * 1e6, scat_dyn / Gamma, lw=2, label="dressed-frame")
axes[2].plot(t_eval * 1e6, scat_eff / Gamma, lw=2, ls="--", label="effective")
axes[2].plot(t_eval * 1e6, scat_static / Gamma, lw=1, alpha=0.6, label="static Ez=200")
axes[2].set_ylabel("scattering rate / Γ")
axes[2].legend(fontsize=10)

axes[3].plot(t_eval * 1e6, photons_dyn, lw=2, label="dressed-frame")
axes[3].plot(t_eval * 1e6, photons_eff, lw=2, ls="--", label="effective")
axes[3].plot(t_eval * 1e6, photons_static, lw=1, alpha=0.6, label="static Ez=200")
axes[3].set_xlabel("t (us)")
axes[3].set_ylabel("cumulative photons")
axes[3].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("\nFinal-state summary:")
print(f"  dressed-frame final excited:  {exc_pop_dyn[-1]:.6e}")
print(f"  effective    final excited:   {exc_pop_eff[-1]:.6e}")
print(f"  dressed-frame total photons:  {photons_dyn[-1]:.6e}")
print(f"  effective    total photons:   {photons_eff[-1]:.6e}")
print(f"  delta photons:                {photons_dyn[-1] - photons_eff[-1]:+.3e}")